In [1]:
import pandas as pd
import numpy as np

statement = pd.read_excel('bank_statement.xlsx')
statement

,Date,Description,Amount
0,2026-01-02,AWS EMEA AMAZON WEB SERVICES,-142.37
1,2026-01-02,GOOGLE *GSUITE acme.io,-18.00
2,2026-01-03,STRIPE TRANSFER ST-X4F2,2400.00
3,2026-01-04,GITHUB.COM 4XJQ2,-21.00
4,2026-01-05,OPENAI *CHATGPT SUBSCR,-20.00
5,2026-01-05,SQ *BLUEBOTTLE 4412,-6.75
6,2026-01-06,WEWORK MEMBERSHIP DUES,-350.00
7,2026-01-07,GUSTO PAYROLL FEE,-45.00
8,2026-01-08,UPWORK ESCROW RELEASE 884213,750.00
9,2026-01-09,AMZN MKTP US*2A4TY8,-63.49


In [2]:
CATEGORY_RULES = {
    "Revenue":["stripe", "upwork", "paypal", "wire credit", "deposit"],
    "Software & SaaS":["aws", "github", "openai", "adobe", "gsuite", "notion", "vercel", "quickbooks"],
    "Payroll":["gusto", "payroll", "salary"],
    "Rent & Office":["wework", "rent", "office lease"],
    "Advertising":["linkedin ads", "google ads", "facebook", "meta ads"],
    "Utilities":["comcast", "verizon", "internet", "wireless"],
    "Travel":["uber trip", "lyft", "delta", "airline", "hotel"],
    "Meals & Entertainment":["doordash", "grubhub", "starbucks", "restaurant", "coffee"],
    "Bank Fees":["service fee", "service charge", "monthly service"],
    "Office Supplies":["staples", "office depot"],
}

In [3]:
def classify(description):
    text = description.lower()
    for category, keywords in CATEGORY_RULES.items():
        if any(keyword in text for keyword in keywords):
            return category
    return "No Match"

In [4]:
statement['Category'] = statement['Description'].apply(classify)
statement

,Date,Description,Amount,Category
0,2026-01-02,AWS EMEA AMAZON WEB SERVICES,-142.37,Software & SaaS
1,2026-01-02,GOOGLE *GSUITE acme.io,-18.00,Software & SaaS
2,2026-01-03,STRIPE TRANSFER ST-X4F2,2400.00,Revenue
3,2026-01-04,GITHUB.COM 4XJQ2,-21.00,Software & SaaS
4,2026-01-05,OPENAI *CHATGPT SUBSCR,-20.00,Software & SaaS
5,2026-01-05,SQ *BLUEBOTTLE 4412,-6.75,No Match
6,2026-01-06,WEWORK MEMBERSHIP DUES,-350.00,Rent & Office
7,2026-01-07,GUSTO PAYROLL FEE,-45.00,Payroll
8,2026-01-08,UPWORK ESCROW RELEASE 884213,750.00,Revenue
9,2026-01-09,AMZN MKTP US*2A4TY8,-63.49,No Match


In [5]:
pnl = statement.groupby('Category')['Amount'].sum()
pnl

Category
Advertising              -120.00
Bank Fees                 -15.00
Meals & Entertainment     -28.50
No Match                 -758.44
Office Supplies           -42.10
Payroll                   -45.00
Rent & Office            -350.00
Revenue                  9650.00
Software & SaaS          -316.36
Travel                   -333.40
Utilities                -174.95
Name: Amount, dtype: float64

In [6]:
net_income = statement['Amount'].sum()
net_income

7466.25

In [7]:
# building clean P&L table
pnl = statement.groupby('Category')['Amount'].sum().sort_values(ascending=False)
pnl_df = pnl.reset_index()
pnl_df.columns = ['Category', 'Amount']

# Adding Net Income row
net_income = statement['Amount'].sum()
pnl_df.loc[len(pnl_df)] = ['NET INCOME', net_income]

# creating excel file
with pd.ExcelWriter('output_report.xlsx', engine='openpyxl') as writer:
    statement.to_excel(writer, sheet_name='Transactions', index=False)
    pnl_df.to_excel(writer, sheet_name='P&L Summary', index=False)
    
print('Saved output_report.xlsx')

Saved output_report.xlsx


In [8]:
from openai import OpenAI
import getpass

client = OpenAI(api_key=getpass.getpass("OpenAI API key: "))

valid_categories = list(CATEGORY_RULES.keys())

def classify_with_ai(description):
    prompt = (
        "Categorize this bank transaction into exactly one of these categories:\n"
        f"{', '.join(valid_categories)}.\n"
        f"Transaction: '{description}'\n"
        "Reply with only the category name, nothing else."
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

mask = statement['Category'] == 'No Match'
statement.loc[mask, 'Category'] = statement.loc[mask, 'Description'].apply(classify_with_ai)

statement.loc[mask, ['Description', 'Category']]

OpenAI API key: ········


,Description,Category
5,SQ *BLUEBOTTLE 4412,Meals & Entertainment
9,AMZN MKTP US*2A4TY8,Office Supplies
10,TST* THE DAILY GRIND,Meals & Entertainment
17,PADDLE.NET* SUPERTOOL,Software & SaaS
25,ZELLE TO CONTRACTOR R LEE,Payroll
28,ACH DEBIT XYZ HOSTING,Software & SaaS


In [9]:
from openpyxl.styles import Font, numbers

# Rebuild the P&L now that the six are reclassified
pnl = statement.groupby('Category')['Amount'].sum().sort_values(ascending=False)
pnl_df = pnl.reset_index()
pnl_df.columns = ['Category', 'Amount']
pnl_df.loc[len(pnl_df)] = ['NET INCOME', statement['Amount'].sum()]

with pd.ExcelWriter('output_report.xlsx', engine='openpyxl') as writer:
    statement.to_excel(writer, sheet_name='Transactions', index=False)
    pnl_df.to_excel(writer, sheet_name='P&L Summary', index=False)

    # Format both sheets
    for sheet in writer.sheets.values():
        # Bold header row
        for cell in sheet[1]:
            cell.font = Font(bold=True)
        # Currency format on the Amount column & auto-size columns
        for col in sheet.columns:
            letter = col[0].column_letter
            width = max(len(str(c.value)) for c in col if c.value is not None) + 3
            sheet.column_dimensions[letter].width = width
            if sheet[f"{letter}1"].value == 'Amount':
                for cell in col[1:]:
                    cell.number_format = '#,##0.00'

    summary = writer.sheets['P&L Summary']
    for cell in summary[summary.max_row]:
        cell.font = Font(bold=True)

print("Saved formatted output_report.xlsx")

Saved formatted output_report.xlsx
